In [0]:
# Cell 1 - Define paths
bronze_table = "workspace.default.bronze_orders"
silver_table = "workspace.default.silver_orders"
checkpoint_path = "/Volumes/workspace/default/raw_orders/_checkpoints/silver"

In [0]:
# Cell 2 - Read from Bronze Delta table as a stream
df_bronze = (
    spark.readStream
    .format("delta")
    .table(bronze_table)
)

In [0]:
# Cell 3 - Clean and transform data
from pyspark.sql import functions as F

df_silver = (
    df_bronze
    # Cast order_date from string to proper date type
    .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))
    # Cast quantity to integer
    .withColumn("quantity", F.col("quantity").cast("integer"))
    # Cast unit_price and total_amount to decimal
    .withColumn("unit_price", F.round(F.col("unit_price").cast("double"), 2))
    .withColumn("total_amount", F.round(F.col("total_amount").cast("double"), 2))
    # Add ingestion timestamp
    .withColumn("ingested_at", F.current_timestamp())
    # Filter out any cancelled orders
    .filter(F.col("status") != "cancelled")
    # Drop duplicates based on order_id
    .dropDuplicates(["order_id"])
)

In [0]:
# Cell 4 - Write stream to Silver Delta table
(
    df_silver.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(silver_table)
)